# Totem Inteligente: Detecção de Faixa Etária com YOLO + DeepFace

**Sprint 4 · Redes Neurais Artificiais, Deep Learning e Algoritmos Genéticos · FIAP 2026**

**Equipe KORA:** Carolina Cordeiro Silva (564234), Gabriel Henrique Pioli (567724), João Victor Tozzatti Matiro (567510), Pedro Diagro Lopes (568393)

---

## O que esse projeto faz

Esse notebook implementa um protótipo de totem inteligente que recebe uma imagem (ou frame de webcam), identifica pessoas, estima a faixa etária de cada uma e gera uma recomendação personalizada em tempo real. Pode ser usado em varejo, museus, eventos ou atendimento automatizado.

## Pipeline

```
Imagem/Webcam
     ↓
  YOLO  (detecta pessoa, retorna bounding box)
     ↓
Recorte da região do rosto
     ↓
DeepFace  (estima idade)
     ↓
Classificação em faixa etária
     ↓
Sistema de recomendação personalizado
     ↓
Output anotado (bbox + idade + faixa + recomendação)
```

## Como rodar

1. Execute as células na ordem.
2. Na seção "Teste com imagem", faça upload de uma foto ou use a URL de exemplo.
3. O output mostra a pessoa detectada, idade estimada e recomendação gerada.

## 1. Instalação das dependências

Executa apenas na primeira vez (ou após reiniciar o runtime).

In [ ]:
!pip install -q ultralytics deepface opencv-python-headless tf-keras
print('OK: dependências instaladas.')

## 2. Imports e configuração

In [ ]:
import os
import cv2
import numpy as np
import urllib.request
from pathlib import Path
import matplotlib.pyplot as plt
from ultralytics import YOLO
from deepface import DeepFace

# Silencia logs do TensorFlow
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Modelo YOLO pequeno e rápido (8MB, suficiente pra detectar pessoa)
MODELO_YOLO = YOLO('yolov8n.pt')

# DeepFace usa 'Age' pra estimar idade (treinado em VGGFace2)
print('OK: YOLO + DeepFace carregados.')

## 3. Definição das faixas etárias e recomendações

Aqui customizamos para o caso de uso do MASP (museu de arte), mas as recomendações podem ser facilmente trocadas para varejo, eventos ou outro contexto.

In [ ]:
# Faixas etárias com limites inclusivos no início, exclusivos no fim
FAIXAS = [
    ('Criança',  0, 12),
    ('Adolescente', 12, 18),
    ('Jovem',    18, 30),
    ('Adulto',   30, 60),
    ('Idoso',    60, 120),
]

# Recomendações por faixa, adaptadas ao caso de uso MASP
RECOMENDACOES = {
    'Criança': {
        'roteiro': 'Quiz Educativo da KORA + Caça ao Tesouro no Acervo Permanente',
        'destaque': 'Cavaletes de cristal de Lina Bo Bardi (visualmente impactantes)',
        'tempo_sugerido': '45 min',
    },
    'Adolescente': {
        'roteiro': 'Exposições contemporâneas + Caça ao Tesouro com cupom no Café',
        'destaque': 'Pop andino (La Chola Poblete), arte urbana atual',
        'tempo_sugerido': '1h',
    },
    'Jovem': {
        'roteiro': 'Histórias Latino-Americanas + Assistente IA pra conversar sobre arte',
        'destaque': 'Réplica (Sandra Gamarra Heshiki), provocações decoloniais',
        'tempo_sugerido': '1h30',
    },
    'Adulto': {
        'roteiro': 'Roteiro personalizado completo + Linha do Tempo MASP',
        'destaque': 'Acervo em Transformação + matéria e energia (Damián Ortega)',
        'tempo_sugerido': '2h',
    },
    'Idoso': {
        'roteiro': 'Acervo Permanente em ritmo tranquilo + Café do MASP',
        'destaque': 'Pinturas europeias clássicas, escolha sentada nos cavaletes',
        'tempo_sugerido': '1h30 com pausa',
    },
}

def classifica_faixa(idade: int) -> str:
    """Mapeia idade numérica em rótulo de faixa etária."""
    for rotulo, minimo, maximo in FAIXAS:
        if minimo <= idade < maximo:
            return rotulo
    return 'Indefinido'

def gera_recomendacao(faixa: str) -> dict:
    """Retorna recomendação curatorial pra faixa etária identificada."""
    return RECOMENDACOES.get(faixa, {
        'roteiro': 'Visita livre',
        'destaque': 'Acervo permanente',
        'tempo_sugerido': '1h',
    })

# Teste rápido
for idade in [8, 15, 25, 45, 70]:
    f = classifica_faixa(idade)
    r = gera_recomendacao(f)
    print(f'{idade} anos → {f}: {r["roteiro"]}')

## 4. Funções da pipeline

Cada etapa é uma função separada, facilitando teste isolado e reuso.

In [ ]:
def detectar_pessoas(image_bgr: np.ndarray) -> list[dict]:
    """
    Detecta pessoas na imagem usando YOLOv8.
    Retorna lista de dicts com bbox (x1,y1,x2,y2) e confiança.
    """
    results = MODELO_YOLO(image_bgr, verbose=False)
    pessoas = []
    for box in results[0].boxes:
        cls = int(box.cls[0])
        if cls == 0:  # classe 0 = 'person' no COCO
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf = float(box.conf[0])
            pessoas.append({'bbox': (x1, y1, x2, y2), 'confianca': conf})
    return pessoas


def estimar_idade(image_bgr: np.ndarray, bbox: tuple) -> int | None:
    """
    Recorta a região da pessoa e estima idade com DeepFace.
    Retorna inteiro (anos) ou None se não conseguir detectar face.
    """
    x1, y1, x2, y2 = bbox
    recorte = image_bgr[max(0, y1):y2, max(0, x1):x2]
    if recorte.size == 0:
        return None
    try:
        resultado = DeepFace.analyze(
            img_path=recorte,
            actions=['age'],
            enforce_detection=False,
            silent=True,
        )
        if isinstance(resultado, list):
            resultado = resultado[0]
        return int(resultado['age'])
    except Exception as e:
        print(f'Falha ao estimar idade: {e}')
        return None


def pipeline_completo(image_bgr: np.ndarray) -> list[dict]:
    """
    Pipeline completo: detecta pessoas, estima idade, classifica faixa, gera recomendação.
    Retorna lista de detecções com todos os campos.
    """
    pessoas = detectar_pessoas(image_bgr)
    saidas = []
    for p in pessoas:
        idade = estimar_idade(image_bgr, p['bbox'])
        if idade is None:
            continue
        faixa = classifica_faixa(idade)
        rec = gera_recomendacao(faixa)
        saidas.append({
            'bbox': p['bbox'],
            'confianca_yolo': p['confianca'],
            'idade_estimada': idade,
            'faixa': faixa,
            'recomendacao': rec,
        })
    return saidas

print('OK: pipeline definida.')

## 5. Função de visualização

Anota a imagem com bounding box, idade, faixa e recomendação.

In [ ]:
def anotar_imagem(image_bgr: np.ndarray, deteccoes: list[dict]) -> np.ndarray:
    """Desenha bbox + texto na imagem."""
    out = image_bgr.copy()
    cor_bbox = (40, 40, 195)  # vermelho cavalete MASP (BGR)
    cor_texto = (255, 255, 255)

    for d in deteccoes:
        x1, y1, x2, y2 = d['bbox']
        cv2.rectangle(out, (x1, y1), (x2, y2), cor_bbox, 3)

        label_top = f"{d['idade_estimada']}a · {d['faixa']}"
        (lw, lh), _ = cv2.getTextSize(label_top, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
        cv2.rectangle(out, (x1, max(0, y1 - lh - 10)), (x1 + lw + 10, y1), cor_bbox, -1)
        cv2.putText(out, label_top, (x1 + 5, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, cor_texto, 2)

        label_rec = d['recomendacao']['roteiro']
        if len(label_rec) > 50:
            label_rec = label_rec[:47] + '...'
        (lw2, lh2), _ = cv2.getTextSize(label_rec, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(out, (x1, y2), (x1 + lw2 + 10, y2 + lh2 + 10), cor_bbox, -1)
        cv2.putText(out, label_rec, (x1 + 5, y2 + lh2 + 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, cor_texto, 1)
    return out


def mostrar(image_bgr: np.ndarray, titulo: str = ''):
    """Exibe imagem BGR (OpenCV) usando matplotlib (que espera RGB)."""
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 8))
    plt.imshow(rgb)
    plt.title(titulo, fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

print('OK: helpers de visualização prontos.')

## 6. Teste com imagem (URL ou upload)

### Opção A · URL de imagem pública (mais rápido)

In [ ]:
URL_IMAGEM = 'https://images.pexels.com/photos/220453/pexels-photo-220453.jpeg?w=800'

urllib.request.urlretrieve(URL_IMAGEM, 'teste.jpg')
imagem = cv2.imread('teste.jpg')
print(f'Imagem carregada: {imagem.shape}')
mostrar(imagem, 'Imagem original')

### Opção B · Upload do seu arquivo (Colab)

In [ ]:
# Descomenta as duas linhas abaixo se quiser fazer upload
# from google.colab import files
# uploaded = files.upload()
# nome_arquivo = list(uploaded.keys())[0]
# imagem = cv2.imread(nome_arquivo)
# mostrar(imagem, f'Imagem enviada: {nome_arquivo}')

## 7. Executar pipeline completa e mostrar resultado

In [ ]:
deteccoes = pipeline_completo(imagem)

print(f'\n=== {len(deteccoes)} pessoa(s) detectada(s) ===\n')
for i, d in enumerate(deteccoes, start=1):
    print(f'Pessoa {i}:')
    print(f'  Bounding box (YOLO): {d["bbox"]} (confiança {d["confianca_yolo"]:.2%})')
    print(f'  Idade estimada (DeepFace): {d["idade_estimada"]} anos')
    print(f'  Faixa etária: {d["faixa"]}')
    print(f'  Recomendação:')
    print(f'     Roteiro:        {d["recomendacao"]["roteiro"]}')
    print(f'     Destaque:       {d["recomendacao"]["destaque"]}')
    print(f'     Tempo sugerido: {d["recomendacao"]["tempo_sugerido"]}')
    print()

if deteccoes:
    imagem_anotada = anotar_imagem(imagem, deteccoes)
    mostrar(imagem_anotada, 'Resultado: pessoa detectada + idade + recomendação')
else:
    print('Nenhuma pessoa detectada na imagem.')

## 8. (Opcional) Webcam ao vivo

Só funciona no Colab via JavaScript ou rodando local. Para gravar o vídeo da demonstração, recomendamos rodar local.

In [ ]:
# Versão webcam ao vivo (rodar LOCAL, fora do Colab):
#
# cap = cv2.VideoCapture(0)
# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break
#     deteccoes = pipeline_completo(frame)
#     frame_anotado = anotar_imagem(frame, deteccoes)
#     cv2.imshow('Totem Faixa Etária', frame_anotado)
#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break
# cap.release()
# cv2.destroyAllWindows()

print('Versão webcam acima, descomente pra rodar local.')

## 9. Conclusão e limitações

### Funcionou?
Sim, a pipeline integrada YOLO + DeepFace funciona em tempo real para imagens estáticas e (rodando local) também em webcam.

### Pontos positivos
- Pipeline modular: cada etapa é função isolada, fácil de testar e trocar
- YOLO leve (8MB) suficiente para detecção de pessoa
- DeepFace usa modelo pré-treinado em VGGFace2, dispensa treino próprio
- Tempo de inferência aceitável (~1s por frame em CPU, mais rápido em GPU)
- Recomendações customizáveis por contexto (museu, varejo, evento)

### Pontos negativos
- Estimativa de idade tem erro médio de ±5 anos (limitação do modelo)
- Iluminação ruim ou ângulo extremo degradam a detecção
- DeepFace pode falhar quando o rosto está parcialmente coberto (máscara, óculos escuros)
- Dependência de hardware com GPU para tempo real fluido
- Pode ter viés do modelo de origem (VGGFace2 sub-representa algumas etnias)

### Melhorias futuras
- Treinar modelo próprio com dataset mais diverso
- Otimizar performance com ONNX Runtime ou TensorRT
- Integrar com o KORA (totem físico) para acionar recomendações automaticamente
- Adicionar detecção de gênero e expressão facial pra refinamento
- Caching de resultado para mesma pessoa em frames consecutivos (evita re-processamento)

---

**Equipe KORA · FIAP 2026 · Sprint 4 Redes Neurais Artificiais, Deep Learning e Algoritmos Genéticos**